In [3]:
# The package named "fitz" on PyPI is NOT PyMuPDF and can trigger:
# "ModuleNotFoundError: No module named 'frontend'".
# PyMuPDF is installed from the "pymupdf" package, but imported as "fitz".

#%pip uninstall -y fitz frontend
%pip install -U pymupdf

import re
import pymupdf
import pandas as pd


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:

PDF_PATH = "data_raw/regulation_docs/FIA 2026 F1 Regulations - Section A [General Regulatory Provisions] - Iss 01 - 2025-12-10.pdf"  # or pymupdf.Document(filename)
MODALS = r"\b(must|shall|may|must not|shall not|prohibited|not permitted|required)\b"
MEASURE = r"(\d+(?:\.\d+)?)\s?(mm|cm|m|kg|g|N|kN|%)\b"

In [20]:
def extract_pages(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({"page": i + 1, "text": text})
    return pages

pages = extract_pages(PDF_PATH)
df = pd.DataFrame(pages)
print(pages[0]["text"][:500])  # Print the first 500 characters of the first page's text

SECTION A: GENERAL REGULATORY PROVISIONS 
 
 
 
0  A 
A1 
2026 Formula 1 Regulations: General Regulatory Provisions 
©2025 Fédération Internationale de l’Automobile 
10 December 2025
Issue 01
SECTION A: GENERAL REGULATORY PROVISIONS 
 
Version: 
 
 
Issue 01 
Status:  
 
 
PUBLISHED 
Date:  
 
 
10/12/2025 
WMSC approval date:  
10/12/2025 
 
CONVENTION: 
Black Text: 
Regulations approved by the WMSC on 10/12/2025 
[Red Text]: 
Information on applicable Governance and relevant Advisory Committee


In [12]:
def clean_text(t: str) -> str:
    # fix hyphenated line breaks: "inter-\nnal" -> "internal"
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)
    # join lines that are broken mid-sentence
    t = re.sub(r"\n+", "\n", t)
    # normalise spaces
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

In [21]:
def split_into_clauses(text: str) -> list[dict]:
    """
    Example clause pattern: 3.1, 3.1.2 etc.
    Tweak this to match how your docs are numbered.
    """
    # Find clause markers
    pattern = re.compile(r"(?m)^(?P<id>\d+(?:\.\d+)+)\s+")
    matches = list(pattern.finditer(text))
    clauses = []

    if not matches:
        return [{"clause_id": None, "clause_text": text.strip()}]

    for idx, m in enumerate(matches):
        start = m.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        clause_id = m.group("id")
        clause_text = text[start:end].strip()
        clauses.append({"clause_id": clause_id, "clause_text": clause_text})
    return clauses

In [23]:
def enrich_clause(clause_text: str) -> dict:
    modals = re.findall(MODALS, clause_text, flags=re.IGNORECASE)
    measures = re.findall(MEASURE, clause_text)
    return {
        "has_constraint_language": bool(modals),
        "modal_terms": sorted(set([m.lower() for m in modals])),
        "measurements": [{"value": v, "unit": u} for v, u in measures],
    }

In [24]:
def build_dataset(pdf_path: str) -> pd.DataFrame:
    pages = extract_pages(pdf_path)
    rows = []

    for p in pages:
        cleaned = clean_text(p["text"])
        for clause in split_into_clauses(cleaned):
            meta = enrich_clause(clause["clause_text"])
            rows.append({
                "page": p["page"],
                "clause_id": clause["clause_id"],
                "text": clause["clause_text"],
                **meta
            })

    return pd.DataFrame(rows)

In [27]:
df = build_dataset(PDF_PATH)
print(df)

     page clause_id                                               text  \
0       1      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
1       2      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
2       3      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
3       4      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
4       5      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
..    ...       ...                                                ...   
114    82       5.5  5.5 \nNo right of appeal \nPU Manufacturers sh...   
115    82       6.1  6.1 \nAn Automotive Manufacturer is a Manufact...   
116    82       6.2  6.2 \nThe Core Activities of an Automotive Man...   
117    83      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
118    84      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   

     has_constraint_language         modal_terms measurements  
0                      False                  [